# 2 · Models: from hand-crafted features to wav2vec2

Three models, all scored on the **same 6 test actors** (360 clips) that no model sees during training:

| model | input | trained by |
|---|---|---|
| Logistic regression | 162 hand-crafted features (ZCR, chroma, 20 MFCCs, RMS, 128 mel bands; mean over time) | `emotion/baselines.py` |
| Original CNN | same 162 features, the original project's Conv1D network (ported to PyTorch) | `emotion/baselines.py` |
| wav2vec2 | raw 16 kHz audio, pretrained `facebook/wav2vec2-base` fine-tuned on 3 s windows | `emotion/finetune.py` |

Training runs as scripts (`uv run python -m emotion.baselines`, `... -m emotion.finetune`) so results don't depend on running
cells in order; this notebook loads and compares what they wrote to `results/`.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from emotion.data import EMOTIONS, RESULTS_DIR

sns.set_theme(style="whitegrid")
metrics = json.loads((RESULTS_DIR / "metrics.json").read_text())
export = json.loads((RESULTS_DIR / "export.json").read_text())

## 1. The original number didn't hold up

The original project reported ~67% accuracy. It augmented every clip (noise, stretch + pitch) *before* a random 80/20 split, so
a test clip's own augmented copies were in the training set, and every test actor was also a training actor. Re-running the
exact same CNN with that protocol and with the speaker-independent one:

In [ ]:
names = {"cnn_leaky": "CNN, original protocol (leaky)", "logreg": "Logistic regression",
         "cnn": "CNN, by-speaker split", "wav2vec2": "wav2vec2 fine-tuned"}
table = pd.DataFrame({names[k]: {"accuracy": m["accuracy"], "macro-F1": m["macro_f1"], "test clips": m["n"]}
                      for k, m in metrics.items() if k in names}).T.loc[list(names.values())]
table.style.format({"accuracy": "{:.1%}", "macro-F1": "{:.1%}"})

The gap between the first and third rows is the leakage: same network, same features, very different number. On unseen
speakers the hand-crafted-feature CNN does not beat a plain logistic regression on the same features by much, which says the
bottleneck is the features (averaged over time, so most of the temporal shape of the speech is gone), not the classifier.

## 2. Per-emotion behaviour

In [ ]:
f1 = pd.DataFrame({names[k]: metrics[k]["per_class_f1"] for k in ("logreg", "cnn", "wav2vec2")}).loc[EMOTIONS]
ax = f1.plot.bar(figsize=(12, 4), rot=0)
ax.set(title="F1 per emotion on unseen speakers", ylabel="F1", ylim=(0, 1));

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, key in zip(axes, ("cnn", "wav2vec2")):
    cm = np.array(metrics[key]["confusion"])
    sns.heatmap(cm / cm.sum(1, keepdims=True), annot=cm, fmt="d", cmap="Blues", cbar=False,
                xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax)
    ax.set(title=names[key], xlabel="predicted", ylabel="actual")
fig.tight_layout()

Reading the matrices (rows are the true emotion, colour is the row-normalised rate):

- The errors that remain are mostly the "neighbour" confusions the EDA predicted: calm ↔ neutral ↔ sad (low arousal) and
  happy ↔ surprise ↔ fear (high arousal).
- Angry is the easiest class for every model: loud, high-pitched and harsh.
- Neutral has half the test clips of the other classes, so its F1 is the noisiest number in the table.

## 3. Why wav2vec2

wav2vec2 was pretrained on 960 h of unlabelled speech to model *how speech sounds over time*. Fine-tuning it on 840 clips reuses
that knowledge instead of learning acoustics from scratch, and it reads the raw waveform, so there is no hand-made feature
pipeline that could drift between training and the app (which was a real bug in the original project: the app computed
different features than training did).

## 4. Shipping it to the browser

The fine-tuned model is exported to ONNX and its matrix multiplications are quantised to int8, so it can run client-side with
transformers.js. `emotion/export.py` checks that the int8 model agrees with PyTorch on unseen speakers before anything ships:

In [ ]:
pd.Series(export).round(3)

## Conclusions

- **Honest evaluation changed the story.** The original 67% came from leakage; on unseen speakers the original CNN is much lower.
- **The pretrained speech model is the real improvement**, mainly on the hard pairs that loudness and pitch can't separate.
- **Remaining limits:** 24 actors, 2 sentences, acted English studio speech. Next steps would be adding more varied data
  (e.g. CREMA-D, which has 91 actors) and evaluating on spontaneous speech.